In [ ]:
"""
═══════════════════════════════════════════════════════════════════
VNDHR Preprocessing Pipeline
Based on: "VNDHR: Variational Single Nighttime Image Dehazing for 
Enhancing Visibility in Intelligent Transportation Systems via 
Hybrid Regularization" (IEEE TITS 2025)

This code preprocesses nighttime hazy images using the VNDHR method
described in the paper. It processes both training and testing datasets.
═══════════════════════════════════════════════════════════════════
"""

import os
import glob
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy import sparse
from scipy.sparse.linalg import cg, LinearOperator

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

# Input directories (raw hazy images)
TRAIN_INPUT = r"D:\Downloads\data\train\input"
TEST_INPUT = r"D:\Downloads\data\test\input"

# Output directories (VNDHR preprocessed)
TRAIN_VNDHR = r"D:\Downloads\data\train_vndhr"
TEST_VNDHR = r"D:\Downloads\data\test_vndhr"

# Create output directories
for directory in [TRAIN_VNDHR, TEST_VNDHR]:
    os.makedirs(directory, exist_ok=True)

# VNDHR Parameters (from paper Section IV-A)
VNDHR_PARAMS = {
    'lambda1': 0.002,    # ℓp norm weight for illumination
    'lambda2': 0.0001,   # Weighted ℓ2 norm for reflectance
    'lambda3': 0.001,    # ℓ1 TV norm for noise suppression
    'p': 0.65,           # Fractional-order norm parameter
    'max_iter': 10,      # Maximum iterations
    'tau1': 1e-6,        # Small constant for gradient magnitude
    'tau2': 1e-6,        # Small constant for gradient magnitude
    'epsilon': 0.001     # Convergence threshold
}

IMG_SIZE = 256  # Resize for computational efficiency

print("═"*80)
print("VNDHR PREPROCESSING PIPELINE")
print("Based on IEEE TITS 2025 Paper")
print("═"*80)
print(f"Parameters: λ1={VNDHR_PARAMS['lambda1']}, λ2={VNDHR_PARAMS['lambda2']}, "
      f"λ3={VNDHR_PARAMS['lambda3']}, p={VNDHR_PARAMS['p']}")
print("═"*80 + "\n")

# ═══════════════════════════════════════════════════════════════════
# GRADIENT OPERATORS (from paper Section III-B)
# ═══════════════════════════════════════════════════════════════════

def gradient_operators(H, W):
    """
    Construct discrete gradient operators using Toeplitz matrices
    Returns: Dx, Dy (sparse matrices for horizontal and vertical gradients)
    """
    N = H * W
    
    # Horizontal gradient (Dx)
    rowi, coli, di = [], [], []
    for r in range(H):
        for c in range(W):
            idx = r * W + c
            if c < W - 1:
                right = r * W + (c + 1)
                rowi.extend([idx, idx])
                coli.extend([idx, right])
                di.extend([-1, 1])
    Dx = sparse.coo_matrix((di, (rowi, coli)), shape=(N, N)).tocsr()
    
    # Vertical gradient (Dy)
    rowi, coli, di = [], [], []
    for r in range(H):
        for c in range(W):
            idx = r * W + c
            if r < H - 1:
                down = (r + 1) * W + c
                rowi.extend([idx, idx])
                coli.extend([idx, down])
                di.extend([-1, 1])
    Dy = sparse.coo_matrix((di, (rowi, coli)), shape=(N, N)).tocsr()
    
    return Dx, Dy

# ═══════════════════════════════════════════════════════════════════
# DIAGONAL PRECONDITIONER (for fast PCG solver)
# ═══════════════════════════════════════════════════════════════════

class DiagonalPreconditioner(LinearOperator):
    """Diagonal preconditioner for conjugate gradient solver"""
    def __init__(self, A):
        self.shape = A.shape
        self.dtype = A.dtype
        self.diag_inv = 1.0 / (A.diagonal() + 1e-8)
    
    def _matvec(self, x):
        return self.diag_inv * x

# ═══════════════════════════════════════════════════════════════════
# ENHANCED VNDHR VARIATIONAL MODEL (Algorithm 1 from paper)
# ═══════════════════════════════════════════════════════════════════

class EnhancedVNDHRVariational:
    """
    Implements the Hybrid Variational Model (HVM) from Section III-B
    Decomposes image into illumination and reflectance using:
    - ℓp norm for structure-aware illumination
    - Weighted ℓ2 norm for fine structures in reflectance  
    - ℓ1 TV norm for noise suppression
    """
    
    def __init__(self, img_np, params=None):
        self.S = img_np.astype(np.float32)
        self.H, self.W = self.S.shape[:2]
        
        # Load parameters
        if params is None:
            params = VNDHR_PARAMS
        self.p = params.get('p', 0.65)
        self.lambda1 = params.get('lambda1', 0.002)
        self.lambda2 = params.get('lambda2', 0.0001)
        self.lambda3 = params.get('lambda3', 0.001)
        self.max_iter = params.get('max_iter', 10)
        self.tau1 = params.get('tau1', 1e-6)
        self.tau2 = params.get('tau2', 1e-6)
        
        # Convert to HSV and extract V-channel (as per paper)
        hsv = cv2.cvtColor((self.S * 255).astype(np.uint8), 
                          cv2.COLOR_RGB2HSV).astype(np.float32) / 255.0
        self.I = hsv[:, :, 2].copy()  # Illumination (V-channel)
        self.R = np.ones_like(self.I)  # Reflectance initialization
        
        # Construct gradient operators
        self.Dx, self.Dy = gradient_operators(self.H, self.W)
    
    def compute_GI(self, I):
        """Compute weight matrix GI for ℓp norm (Eq. 8)"""
        gradIx = self.Dx.dot(I.flatten()).reshape(self.H, self.W)
        gradIy = self.Dy.dot(I.flatten()).reshape(self.H, self.W)
        magI = np.sqrt(gradIx**2 + gradIy**2)
        return np.power(np.maximum(magI, self.tau1), self.p - 2)
    
    def compute_WR(self, R):
        """Compute weight matrix WR for weighted ℓ2 norm (Eq. 6)"""
        gradRx = self.Dx.dot(R.flatten()).reshape(self.H, self.W)
        gradRy = self.Dy.dot(R.flatten()).reshape(self.H, self.W)
        magR = np.sqrt(gradRx**2 + gradRy**2)
        return 1.0 / (np.maximum(magR, self.tau2) + 1e-8)
    
    def compute_GR(self, R):
        """Compute weight matrix GR for ℓ1 norm (Eq. 9)"""
        gradRx = self.Dx.dot(R.flatten()).reshape(self.H, self.W)
        gradRy = self.Dy.dot(R.flatten()).reshape(self.H, self.W)
        magR = np.sqrt(gradRx**2 + gradRy**2)
        return np.maximum(magR, self.tau2) ** (-1)
    
    def solve_I_subproblem(self, R, GI):
        """Solve I sub-problem using PCG (Eq. 13)"""
        Rdiag = sparse.diags(R.flatten())
        Wgi = sparse.diags(GI.flatten())
        U = self.Dx.T.dot(Wgi.dot(self.Dx)) + self.Dy.T.dot(Wgi.dot(self.Dy))
        
        A_I = Rdiag.T.dot(Rdiag) + self.lambda1 * U
        b_I = Rdiag.T.dot(self.I.flatten())
        
        M = DiagonalPreconditioner(A_I)
        x0 = self.I.flatten()
        
        x_sol, info = cg(A_I, b_I, x0=x0, M=M, atol=1e-5, rtol=1e-5, maxiter=300)
        if info != 0:
            x_sol = x0
        
        return x_sol.reshape(self.H, self.W)
    
    def solve_R_subproblem(self, I, WR, GR):
        """Solve R sub-problem using PCG (Eq. 17)"""
        Idiag = sparse.diags(I.flatten())
        Wwr = sparse.diags(WR.flatten())
        Wgr = sparse.diags(GR.flatten())
        
        V = self.Dx.T.dot(Wwr.dot(self.Dx)) + self.Dy.T.dot(Wwr.dot(self.Dy))
        M_term = self.Dx.T.dot(Wgr.dot(self.Dx)) + self.Dy.T.dot(Wgr.dot(self.Dy))
        
        LHS_R = Idiag.T.dot(Idiag) + self.lambda2 * V + self.lambda3 * M_term
        b_R = Idiag.T.dot(I.flatten())
        
        M_prec = DiagonalPreconditioner(LHS_R)
        r0 = self.R.flatten()
        
        r_sol, info = cg(LHS_R, b_R, x0=r0, M=M_prec, atol=1e-5, rtol=1e-5, maxiter=300)
        if info != 0:
            r_sol = r0
        
        return np.clip(r_sol.reshape(self.H, self.W), 0, 2.5)
    
    def run(self):
        """Run iterative optimization (Algorithm 1)"""
        for iteration in range(self.max_iter):
            # Compute weight matrices
            GI = self.compute_GI(self.I)
            WR = self.compute_WR(self.R)
            GR = self.compute_GR(self.R)
            
            # Update I and R
            I_new = self.solve_I_subproblem(self.R, GI)
            R_new = self.solve_R_subproblem(I_new, WR, GR)
            
            # Check convergence
            if (np.linalg.norm(I_new - self.I) / np.linalg.norm(self.I) < 1e-3 and
                np.linalg.norm(R_new - self.R) / np.linalg.norm(self.R) < 1e-3):
                break
            
            self.I = I_new
            self.R = R_new
        
        return self.I, self.R

# ═══════════════════════════════════════════════════════════════════
# PREPROCESSING FUNCTION
# ═══════════════════════════════════════════════════════════════════

def process_vndhr_folder(input_dir, output_dir, img_size=IMG_SIZE):
    """
    Process all images in a folder using VNDHR
    
    Args:
        input_dir: Directory containing input hazy images
        output_dir: Directory to save VNDHR preprocessed results
        img_size: Resize dimension for processing
    """
    files = sorted(glob.glob(os.path.join(input_dir, "*.*")))
    
    if len(files) == 0:
        print(f"⚠️  No images found in {input_dir}")
        return 0
    
    print(f"Processing {len(files)} images from {os.path.basename(input_dir)}")
    success_count = 0
    
    for path in tqdm(files, desc=f"VNDHR → {os.path.basename(output_dir)}"):
        try:
            # Load and resize image
            img = Image.open(path).convert("RGB")
            img = img.resize((img_size, img_size), Image.LANCZOS)
            img_np = np.array(img) / 255.0
            
            # Apply VNDHR decomposition
            vndhr = EnhancedVNDHRVariational(img_np, params=VNDHR_PARAMS)
            I, R = vndhr.run()
            
            # Reconstruct VNDHR output: S' = I ◦ R
            out = np.clip(img_np * I[:, :, None] * R[:, :, None], 0, 1)
            
            # Save result
            base = os.path.basename(path)
            save_path = os.path.join(output_dir, base)
            Image.fromarray((out * 255).astype(np.uint8)).save(save_path, quality=95)
            
            success_count += 1
            
        except Exception as e:
            print(f"\n⚠️  Error processing {os.path.basename(path)}: {e}")
    
    print(f"✓ Successfully processed {success_count}/{len(files)} images\n")
    return success_count

# ═══════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("\n" + "═"*80)
    print("STEP 1: Processing Training Data")
    print("═"*80)
    train_count = process_vndhr_folder(TRAIN_INPUT, TRAIN_VNDHR, IMG_SIZE)
    
    print("═"*80)
    print("STEP 2: Processing Testing Data")
    print("═"*80)
    test_count = process_vndhr_folder(TEST_INPUT, TEST_VNDHR, IMG_SIZE)
    
    print("═"*80)
    print("VNDHR PREPROCESSING COMPLETE")
    print("═"*80)
    print(f"Training images processed: {train_count}")
    print(f"Testing images processed:  {test_count}")
    print(f"\nOutput directories:")
    print(f"  Train: {TRAIN_VNDHR}")
    print(f"  Test:  {TEST_VNDHR}")
    print("═"*80 + "\n")

Using device: cuda
Created/verified directory: D:\Downloads\data\train_vndhr
Created/verified directory: D:\Downloads\data\test_vndhr
Created/verified directory: D:\Downloads\data\test\dehazed_results
Created/verified directory: D:\Downloads\data\test\comparison_results
High-Quality VNDHR + UNet Dehazing Pipeline
Target: PSNR 28+, SSIM 0.80+

=== Dataset Validation ===
✓ Train Input: 5000 files found
  Sample files: ['1.jpg', '10.jpg', '100.jpg']
✓ Train Target: 5000 files found
  Sample files: ['1.jpg', '10.jpg', '100.jpg']
✓ Test Input: 1001 files found
  Sample files: ['0001_0.8_0.2.jpg', '0002_0.8_0.08.jpg', '0003_0.8_0.2.jpg']
✓ Test Target: 1001 files found
  Sample files: ['0001_0.8_0.2.jpg', '0002_0.8_0.08.jpg', '0003_0.8_0.2.jpg']

Matching train pairs: 5000


STEP 1: Enhanced VNDHR Preprocessing

Processing 5000 images with Enhanced VNDHR


VNDHR -> train_vndhr: 100%|█████████████████████████████████████████████████████| 5000/5000 [11:44:24<00:00,  8.45s/it]


Successfully processed 5000/5000 images

Processing 1001 images with Enhanced VNDHR


VNDHR -> test_vndhr: 100%|███████████████████████████████████████████████████████| 1001/1001 [2:11:24<00:00,  7.88s/it]

Error loading D:\Downloads\data\test\input\Thumbs.db: cannot identify image file 'D:\\Downloads\\data\\test\\input\\Thumbs.db'

Error processing Thumbs.db: cannot identify image file 'D:\\Downloads\\data\\test\\input\\Thumbs.db'
Successfully processed 1000/1001 images

STEP 2: Enhanced UNet Training (High Quality Mode)
Dataset created: 5000 pairs from 5000 VNDHR images



C:\Users\karth\AppData\Local\Temp\ipykernel_5400\2566242959.py:389: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if ENABLE_AMP and device.type == 'cuda' else None



Starting training: 5000 samples, 625 batches/epoch
Mixed Precision: Enabled


Epoch 1/30:   0%|                                                                              | 0/625 [00:00<?, ?it/s]

In [9]:
# -----------------------------
# Perceptual Loss Training for VNDHR (Target: PSNR 28+, SSIM 0.8+)
# Uses VGG perceptual loss instead of MS-SSIM
# -----------------------------
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
import os, glob
from PIL import Image
import torch.nn as nn
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity as compare_ssim
import numpy as np

# --- Paths ---
VNDHR_TRAIN = r"D:\Downloads\data\train_vndhr"
TRAIN_TARGET = r"D:\Downloads\data\train\target"
MODEL_PATH = r"D:\Downloads\data\unet_vndhr_fixed.pth"
NEW_MODEL_PATH = r"D:\Downloads\data\unet_vndhr_perceptual.pth"

# --- Hyperparameters ---
IMG_SIZE = 256
BATCH_SIZE = 4
TOTAL_EPOCHS = 60
START_EPOCH = 10
LR = 3e-5  # Conservative LR
NUM_WORKERS = 0
ENABLE_AMP = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --- Perceptual Loss (VGG-based) ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True).features[:16].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg.to(device)
        self.mse = nn.MSELoss()
        
    def forward(self, pred, target):
        pred_feat = self.vgg(pred)
        target_feat = self.vgg(target)
        return self.mse(pred_feat, target_feat)

# --- Dataset ---
class VNDHRDataset(Dataset):
    def __init__(self, vndhr_dir, clear_dir, img_size=IMG_SIZE, augment=True):
        v_files = sorted(glob.glob(os.path.join(vndhr_dir, "*.*")))
        c_files = sorted(glob.glob(os.path.join(clear_dir, "*.*")))
        c_dict = {os.path.basename(f): f for f in c_files}
        self.pairs = [(v, c_dict[os.path.basename(v)]) for v in v_files if os.path.basename(v) in c_dict]
        self.img_size = img_size
        self.augment = augment
        print(f"Dataset: {len(self.pairs)} pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        v_path, c_path = self.pairs[idx]
        v_img = Image.open(v_path).convert("RGB").resize((self.img_size, self.img_size), Image.LANCZOS)
        c_img = Image.open(c_path).convert("RGB").resize((self.img_size, self.img_size), Image.LANCZOS)

        if self.augment:
            if np.random.rand() > 0.5:
                v_img, c_img = TF.hflip(v_img), TF.hflip(c_img)
            if np.random.rand() > 0.5:
                v_img, c_img = TF.vflip(v_img), TF.vflip(c_img)
            if np.random.rand() > 0.5:
                angle = int(np.random.choice([90, 180, 270]))
                v_img, c_img = TF.rotate(v_img, angle), TF.rotate(c_img, angle)

        return transforms.ToTensor()(v_img), transforms.ToTensor()(c_img)

# --- UNet ---
class ImprovedUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, base_c=64):
        super().__init__()
        self.enc1 = self._block(in_ch, base_c)
        self.enc2 = self._block(base_c, base_c*2)
        self.enc3 = self._block(base_c*2, base_c*4)
        self.enc4 = self._block(base_c*4, base_c*8)
        self.bottleneck = self._block(base_c*8, base_c*16)
        self.up4 = nn.ConvTranspose2d(base_c*16, base_c*8, 2, stride=2)
        self.dec4 = self._block(base_c*16, base_c*8)
        self.up3 = nn.ConvTranspose2d(base_c*8, base_c*4, 2, stride=2)
        self.dec3 = self._block(base_c*8, base_c*4)
        self.up2 = nn.ConvTranspose2d(base_c*4, base_c*2, 2, stride=2)
        self.dec2 = self._block(base_c*4, base_c*2)
        self.up1 = nn.ConvTranspose2d(base_c*2, base_c, 2, stride=2)
        self.dec1 = self._block(base_c*2, base_c)
        self.final_conv = nn.Conv2d(base_c, out_ch, 1)
        self.pool = nn.MaxPool2d(2)
        
    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.final_conv(d1))

# --- DataLoader ---
dataset = VNDHRDataset(VNDHR_TRAIN, TRAIN_TARGET, IMG_SIZE, augment=True)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

# --- Load ORIGINAL Epoch 10 Checkpoint (22.29 dB) ---
model = ImprovedUNet(base_c=64).to(device)
checkpoint = torch.load(MODEL_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
best_psnr = checkpoint['psnr']
print(f"✓ Restored checkpoint from epoch {checkpoint['epoch']} | PSNR: {best_psnr:.2f} dB")

# --- Optimizer & Losses ---
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

mse_loss = nn.MSELoss()
l1_loss = nn.L1Loss()
perceptual_loss = VGGPerceptualLoss()
scaler = torch.amp.GradScaler(enabled=ENABLE_AMP)

patience = 0
max_patience = 15

print(f"\n🚀 Training with Perceptual Loss (VGG-based)")
print(f"Starting PSNR: {best_psnr:.2f} dB | Target: 28+ dB\n")

# --- Training Loop ---
for epoch in range(START_EPOCH, TOTAL_EPOCHS):
    model.train()
    epoch_loss = 0
    psnr_list, ssim_list = [], []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}")

    # Gradually introduce perceptual loss
    if epoch < 20:
        w_pixel = 1.0
        w_perc = 0.0
    elif epoch < 40:
        progress = (epoch - 20) / 20
        w_pixel = 1.0
        w_perc = 0.1 * progress  # Max 0.1
    else:
        w_pixel = 1.0
        w_perc = 0.1

    for v, c in pbar:
        v, c = v.to(device).float(), c.to(device).float()
        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=ENABLE_AMP):
            pred = model(v)
            
            # Pixel-wise losses
            loss_mse = mse_loss(pred, c)
            loss_l1 = l1_loss(pred, c)
            
            # Perceptual loss (disabled during AMP for stability)
            if w_perc > 0:
                with torch.cuda.amp.autocast(enabled=False):
                    loss_perc = perceptual_loss(pred.float(), c.float())
                loss = w_pixel * (0.5*loss_mse + 0.5*loss_l1) + w_perc * loss_perc
            else:
                loss = 0.5*loss_mse + 0.5*loss_l1

        if torch.isnan(loss) or torch.isinf(loss):
            print("⚠️ Invalid loss, skipping")
            continue

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

        # Evaluation
        with torch.no_grad():
            pred_np = (pred.clamp(0,1).cpu().numpy()*255).astype(np.uint8)
            c_np = (c.cpu().numpy()*255).astype(np.uint8)
            for i in range(pred_np.shape[0]):
                psnr_list.append(compare_psnr(c_np[i].transpose(1,2,0), pred_np[i].transpose(1,2,0), data_range=255))
                ssim_list.append(compare_ssim(c_np[i].transpose(1,2,0), pred_np[i].transpose(1,2,0), channel_axis=2, data_range=255))

        pbar.set_postfix(loss=f"{loss.item():.4f}", psnr=f"{np.mean(psnr_list):.2f}", ssim=f"{np.mean(ssim_list):.3f}")

    avg_psnr = np.mean(psnr_list)
    avg_ssim = np.mean(ssim_list)
    avg_loss = epoch_loss / len(dataloader)
    
    print(f"\nEpoch {epoch+1} | Loss: {avg_loss:.4f} | PSNR: {avg_psnr:.2f} dB | SSIM: {avg_ssim:.4f}")
    print(f"Weights - Pixel: {w_pixel:.2f}, Perceptual: {w_perc:.2f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
    
    scheduler.step(avg_psnr)

    # Save best model
    if avg_psnr > best_psnr:
        best_psnr = avg_psnr
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'psnr': avg_psnr,
            'ssim': avg_ssim
        }, NEW_MODEL_PATH)
        print(f"✓ Best model saved! PSNR: {avg_psnr:.2f} dB, SSIM: {avg_ssim:.4f}")
        patience = 0
    else:
        patience += 1
        
    if patience >= max_patience:
        print(f"\n⚠️ Early stopping at epoch {epoch+1}")
        break
    if avg_psnr >= 28.0 and avg_ssim >= 0.8:
        print(f"\n🎯 TARGET REACHED! PSNR: {avg_psnr:.2f} dB, SSIM: {avg_ssim:.4f}")
        break

print(f"\n✓ Training complete. Best PSNR: {best_psnr:.2f} dB")
print(f"Model saved to {NEW_MODEL_PATH}")

Using device: cuda
Dataset: 5000 pairs
✓ Restored checkpoint from epoch 9 | PSNR: 22.29 dB


C:\Users\karth\medgrad_env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\karth\medgrad_env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\karth/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100%|███████████████████████████████████████████████████████████████████████████████| 528M/528M [00:44<00:00, 12.3MB/s]



🚀 Training with Perceptual Loss (VGG-based)
Starting PSNR: 22.29 dB | Target: 28+ dB



...


Epoch 58 | Loss: 0.0787 | PSNR: 23.88 dB | SSIM: 0.8552
Weights - Pixel: 1.00, Perceptual: 0.10 | LR: 3.00e-05
✓ Best model saved! PSNR: 23.88 dB, SSIM: 0.8552


Epoch 59/60: 100%|████████████████████████████| 1250/1250 [12:15<00:00,  1.70it/s, loss=0.0658, psnr=23.95, ssim=0.856]



Epoch 59 | Loss: 0.0780 | PSNR: 23.95 dB | SSIM: 0.8559
Weights - Pixel: 1.00, Perceptual: 0.10 | LR: 3.00e-05
✓ Best model saved! PSNR: 23.95 dB, SSIM: 0.8559


Epoch 60/60: 100%|████████████████████████████| 1250/1250 [12:12<00:00,  1.71it/s, loss=0.0753, psnr=24.01, ssim=0.857]



Epoch 60 | Loss: 0.0776 | PSNR: 24.01 dB | SSIM: 0.8567
Weights - Pixel: 1.00, Perceptual: 0.10 | LR: 3.00e-05
✓ Best model saved! PSNR: 24.01 dB, SSIM: 0.8567

✓ Training complete. Best PSNR: 24.01 dB
Model saved to D:\Downloads\data\unet_vndhr_perceptual.pth
